Detección de caras con webcam con detectores de OpenCV

In [1]:
import cv2
import numpy as np
from mtcnn.mtcnn import MTCNN

# Carga de modelos
# VJ. Cara y Ojos

# Enlace a todos los Haar cascades dentro de OpenCV: https://github.com/opencv/opencv/tree/4.x/data/haarcascades
# Hay algunos muy curiosos, frontalcatface? russian_plate_number???

haar_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
#eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

#Este parece que va mejor con gafas.

eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye_tree_eyeglasses.xml')


# DNN
dnn_model = "deploy.prototxt.txt"
dnn_weights = "res10_300x300_ssd_iter_140000.caffemodel"
net = cv2.dnn.readNetFromCaffe(dnn_model, dnn_weights)

# MTCNN
mtcnn_det = MTCNN()

# Un par de funciones para la visualización. 
def clip_boxes(boxes, frame_w, frame_h):
    clipped = []
    for (x, y, w, h) in boxes:
        x1, y1 = max(int(x), 0), max(int(y), 0)
        x2, y2 = min(int(x + w), frame_w), min(int(y + h), frame_h)
        if x2 - x1 > 0 and y2 - y1 > 0:
            clipped.append((x1, y1, x2 - x1, y2 - y1))
    return clipped

def draw_mtcnn_keypoints(frame, keypoints_list):
    # keypoints_list: una entrada por cara, cada una es un diccionario
    # {"left_eye": (x, y), "right_eye": (x, y), "nose": ..., "mouth_left": ..., "mouth_right": ...}
    for keypoints in keypoints_list:
        for _, (px, py) in keypoints.items():
            cv2.circle(frame, (int(px), int(py)), 4,(0, 255, 255), -1)

def draw_eyes(frame, gray, boxes):
    # Detecta los ojos dentro de las caras usando el Haar cascade de ojos.
    for (x, y, w, h) in clip_boxes(boxes, frame.shape[1], frame.shape[0]):
        
        roi = gray[y:y + h, x:x + w]
        eyes = eye_cascade.detectMultiScale(roi, scaleFactor=1.1, minNeighbors=5, minSize=(15, 15))
        for (ex, ey, ew, eh) in eyes:
        # Dibujamos un circulo amarillo en el centro de cada ojo detectado
            cv2.circle(frame, (x + ex + ew // 2, y + ey + eh // 2), 4, (0, 255, 255), -1)


video_capture = cv2.VideoCapture(0)


det = 0
show_landmarks = False

while True:
    ret, frame = video_capture.read()
    if not ret:
        break

    # Pasamos a escala de grises el frame para el detector Haar y para la detección de ojos
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    H, W = frame.shape[:2]

    boxes = []           
    mtcnn_points = []

    
    if det == 0:
        name = "VJ"
        # VJ
        faces = haar_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

        
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
            boxes.append((x, y, w, h))
    elif det == 1:
        name = "DNN"
        # DNN
        blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), [104.0, 117.0, 123.0], False, False)
        net.setInput(blob)
        detections = net.forward()

        
        for i in range(detections.shape[2]):
            confidence = detections[0, 0, i, 2]
            if confidence > 0.5:
                box = detections[0, 0, i, 3:7] * np.array([W, H, W, H])
                (x, y, x2, y2) = box.astype("int")
                cv2.rectangle(frame, (x, y), (x2, y2), (0, 255, 0), 2)
                boxes.append((x, y, x2 - x, y2 - y))
    else: 
        name = "MTCNN"
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = mtcnn_det.detect_faces(rgb_frame)
        
        # Draw bounding boxes
        for result in results:
            x, y, width, height = result['box']
            cv2.rectangle(frame, (x, y), (x + width, y + height), (0, 0, 255), 2)

        mtcnn_points.append(result['keypoints'])
    if show_landmarks:
        if det in (0, 1):
            # El detector de ojos de Haar se aplica solo si no estamos usando MTCNN, este tiene uno integrado.
            draw_eyes(frame, gray, boxes)
        elif mtcnn_points is not None:
            # MTCNN devuelve 5 puntos de referencia (Ojos, nariz y lados de la boca)
            draw_mtcnn_keypoints(frame, mtcnn_points)

    # Mostramos resultados
    cv2.putText(frame, f"Detector: {name}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 255), 2)
    cv2.putText(frame, "Presiona 'd' para cambiar de detector, 'e' para mostrar landmarks, Esc para salir",
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
    if not show_landmarks:
        status = "Eyes/landmarks: OFF"
    elif det == 2:
        status = "Landmarks: ON (MTCNN, 5 puntos)"
    else:
        status = "Eyes: ON (Haar eye cascade)"
    cv2.putText(frame, status, (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    cv2.imshow('Face Detection', frame)

    # 'd' para cambiar de detector, 'e' para mostrar landmarks, Esc para salir"
    key = cv2.waitKey(1) & 0xFF
    if key == ord('d'):
        det = (det + 1) % 3
    elif key == ord('e'):
        show_landmarks = not show_landmarks
    elif key == 27:
        break

# Cerramos
video_capture.release()
cv2.destroyAllWindows()

